# 02 — External validation on MIMIC-III

**Before you start**

* Sign in to Colab with the Google account that PhysioNet granted MIMIC-III BigQuery access
  to (shown on the MIMIC-III page: *"Access to the GCP BigQuery was previously awarded to ..."*),
  and choose the same account in every sign-in window below.
* You need a Google Cloud project ID that this account can run BigQuery jobs in
  (console.cloud.google.com, project selector at the top). Put it in `GCP_PROJECT` below.

Run the cells in order. Never commit or share `data/mimic_cache/` or the EF annotation sheet
(PhysioNet Data Use Agreement); the `results/tables/mimic_*.csv` files are aggregate and
can be shared.

In [ ]:
GCP_PROJECT = 'physionet-data-491418'   # <- your Google Cloud project ID

In [ ]:
# Setup (Colab or local). Outputs go to HF_PROJECT_ROOT, a Google Drive folder on Colab,
# so they survive a disconnect. Re-running this cell is safe.
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = '/content/HF-Survival-Analysis-Leakage-Revisit'
def sh(*cmd):
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError('Command failed: ' + ' '.join(cmd))
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_PROJECT_ROOT'] = '/content/drive/MyDrive/HF_Revisit_Project'
    if not os.path.exists(REPO_DIR):
        sh('git', 'clone', '-q', '-b', 'claude/ecstatic-wright-61dz0b', 'https://github.com/elifnuroksuzz/HF-Survival-Analysis-Leakage-Revisit.git', REPO_DIR)
    os.chdir(REPO_DIR)
    sh('git', 'fetch', '-q', 'origin', 'claude/ecstatic-wright-61dz0b')
    sh('git', 'reset', '-q', '--hard', 'origin/claude/ecstatic-wright-61dz0b')   # always the latest code
    sh(sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[mimic]')
sys.path.insert(0, 'src')
from hfrevisit import config as C
P = C.get_paths()
print('Code version:', subprocess.run(['git', 'log', '-1', '--format=%h %cd', '--date=short'],
                                      capture_output=True, text=True).stdout.strip())
print('Outputs are written to:', P.root)
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print('Signed in.')

In [ ]:
# Access check (a few seconds): fails early, with a clear message, if the account or project is wrong.
import pandas_gbq
try:
    n = pandas_gbq.read_gbq('SELECT COUNT(*) AS n FROM `physionet-data.mimiciii_clinical.patients`',
                            project_id=GCP_PROJECT, progress_bar_type=None, location='US')['n'][0]
    print(f'MIMIC-III access OK: {n:,} patients.')
except Exception as exc:
    raise RuntimeError(
        'No access to MIMIC-III on BigQuery.\n'
        ' - "Access Denied" / "permission": you signed in with a different Google account than the one\n'
        '   PhysioNet authorised. Runtime > Disconnect and delete runtime, then run again with that account.\n'
        ' - "Project ... not found" / "bigquery.jobs.create": GCP_PROJECT is wrong or belongs to another\n'
        '   account; copy the project ID shown at console.cloud.google.com (same account).\n'
        f'Original error: {exc}') from None

In [ ]:
# Step 1: train and save the development models on the public data (~2 min)
!python scripts/run_development.py --download --stages export

In [ ]:
# Step 2: external validation (BigQuery extraction ~5-15 min the first time, then cached; analysis ~10-30 min)
import runpy
sys.argv = ['run_external_validation.py', '--gcp-project', GCP_PROJECT, '--horizon', '90']
runpy.run_path('scripts/run_external_validation.py', run_name='__main__')

In [ ]:
# Step 3: draw 200 echo reports for the manual EF check (sheet stays on your Drive; do not share it)
!python scripts/ef_annotation.py sample

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_columns', 40)
def table(name): return pd.read_csv(P.tables / f'{name}.csv')
def figure(name): display(Image(filename=str(P.figures / f'{name}.png'), width=900))

## Results

In [ ]:
table('mimic_primary_attrition')

In [ ]:
table('mimic_primary_performance').round(3)

In [ ]:
for s in ['sens_numeric_ef', 'sens_complete_cpk', 'sens_age40']:
    print(s); display(table(f'mimic_{s}_performance').round(3))

In [ ]:
figure('fig5_mimic_calibration')
table('mimic_primary_recalibration').round(3)

In [ ]:
display(table('mimic_primary_subgroups').round(3)); table('mimic_primary_subgroup_gaps').round(3)

## Text to send back (aggregate results only)
Copy everything this cell prints and paste it into the chat.

In [ ]:
import glob
for f in sorted(glob.glob(str(P.tables / 'mimic_*.csv'))):
    d = pd.read_csv(f)
    if 'subgroups' in f:                       # small-cell suppression (PhysioNet guidance)
        d = d[d['n'] >= 10].astype({'events': object})
        d.loc[d['events'] < 10, 'events'] = '<10'
    print('=====', os.path.basename(f)); print(d.round(4).to_csv(index=False))

## Later: score the EF annotation
Open `HF_Revisit_Project/data/mimic_cache/ef_annotation_sheet.csv` on Drive. You and a colleague
independently type the EF stated in each report into `ef_manual_a` and `ef_manual_b`
(a number, or `NA` if no EF is stated). Then run the cell below.

In [ ]:
!python scripts/ef_annotation.py score